# RAG workshop — День 2: асистент ріелтора (realistic end-to-end)

Цей ноутбук повторює ланцюг **завантаження → chunking → embeddings → векторний індекс → retrieval → промпт → відповідь**, але на **реалістичному** (хоча й синтетичному) **корпусі** з папки `data/`: каталог оголошень (CSV), клієнти/угоди (CSV), політики та чеклисти (Markdown),  договори у **`data/contracts/`** (PDF).

**Сьогодні ми розберемо:**
- Реалістичні файли документів різних форматів замість окремих рядків прямо в коді;
- **ChromaDB у режимі персистентності** (`PersistentClient`) — індекс зберігається між перезапусками;
- Створення **метаданих** для обробленних: поле **`source`** (шлях до файлу);
- **Стабільна індексація** через `upsert` і стабільні `id` (повторний запуск не дублює вектори);
- **Промпт інжінрінг** - відповідь з **посиланням на джерела** (які файли/записи потрапили в контекст). Детальний роутінг до конкретного файлу. 
- **Чат-бот** через Телеграм, для імітації інтерфейсу аплікації 

Контекст юзкейсу: `use_case_realtor_assistant.md`.



### Примітка щодо Colab

Цей ноутбук працює як на **Google Colab**, так і на **локальній машині**:

**У Google Colab:**
- 📂 **Автоматичне завантаження**: папка `data/` і ресурси завантажуються з GitHub (не потрібно вручну завантажувати файли на Drive!)
- 🔑 Додайте secrets через ліву панель (🔑 key icon):
  - `OPENAI_API_KEY` — ваш OpenAI ключ
  - `TELEGRAM_TOKEN` — (опційно) для Telegram-бота
- ▶️ Запустіть комірки послідовно (Shift+Enter)
- 💾 Результати (вектор-індекс) зберігаються в `/content` (тимчасово для сесії)

**Локально:**
- Клонуйте repo
- Створіть `venv`: `python -m venv .venv && source .venv/bin/activate`
- Встановіть залежності: `pip install -r requirements.txt`
- Додайте `.env` з credentials: `OPENAI_API_KEY=...`, `TELEGRAM_TOKEN=...`
- Запустіть ноутбук: `jupyter notebook`

## Що потрібно

- `OPENAI_API_KEY` 
- `TELEGRAM_TOKEN`



## 00 - Setup
- Встановлюємо бібліотеки
- Считуємо доступи, паролі і шляхі до даних 
- Створюємо допоміжні функції 

In [19]:
# Встановлюємо залежності
import os
import sys

# Colab detection
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🔧 Colab detected. Installing packages...")
    # Встановлюємо бібліотеки в Colab
    !pip install openai chromadb python-dotenv pandas pymupdf python-telegram-bot -q
else:
    print("💻 Local environment detected.")
    !pip install openai chromadb python-dotenv pandas pymupdf python-telegram-bot -q

💻 Local environment detected.


In [ ]:
# === Завантаження файлів з GitHub для Colab ===
from pathlib import Path
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("📥 Downloading project files from GitHub...")
    import subprocess
    project_root = Path('/content/RAG-workshop_Day2')
    
    # Клонуємо репозиторій у /content (без Google Drive)
    try:
        subprocess.run(
            ["rm", "-rf", str(project_root)],
            capture_output=True
        )
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/NatalyUA/RAG-workshop_Day2.git",
             str(project_root)],
            check=True,
            capture_output=True,
            text=True,
        )
        print("✓ Repository cloned")
    except Exception as e:
        print(f"⚠️  Could not clone repository: {e}")

    data_dir = project_root / 'data'
    print(f"data exists: {data_dir.is_dir()}")
    
    print("✅ Files ready for processing")


In [ ]:
import csv
import os
import re
import sys
from pathlib import Path

import chromadb
import fitz  # PyMuPDF — читання PDF
from dotenv import load_dotenv
from openai import OpenAI

# --- Colab detection ---
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import userdata
    PROJECT_ROOT = Path('/content/RAG-workshop_Day2')
    print(f"📂 Using project root in Colab: {PROJECT_ROOT}")
    if not (PROJECT_ROOT / 'data').is_dir():
        raise FileNotFoundError(
            "data/ не знайдено. Запустіть попередню комірку: Downloading project files from GitHub."
        )
else:
    PROJECT_ROOT = Path.cwd()
    for ancestor in [Path.cwd(), *Path.cwd().parents]:
        if (ancestor / "data" / "listings.csv").is_file():
            PROJECT_ROOT = ancestor
            break

# --- Load credentials ---
if IN_COLAB:
    # На Colab: читаємо з Secrets (через userdata.get)
    try:
        os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
        print("✓ OPENAI_API_KEY loaded from Colab Secrets")
    except Exception as e:
        print(f"⚠️  Could not load OPENAI_API_KEY from Secrets: {e}")
    
    try:
        os.environ["TELEGRAM_TOKEN"] = userdata.get('TELEGRAM_TOKEN')
        print("✓ TELEGRAM_TOKEN loaded from Colab Secrets")
    except:
        print("⚠️  TELEGRAM_TOKEN not found (Telegram bot won't work)")
else:
    # На локальній машині: читаємо з .env
    _env_loaded = False
    for p in [PROJECT_ROOT / ".env", *Path.cwd().parents]:
        p = Path(p)
        if p.is_file() and p.name == ".env":
            load_dotenv(p)
            _env_loaded = True
            break
    if not _env_loaded:
        load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "Немає OPENAI_API_KEY. На Colab: додайте до Secrets (🔑). Локально: додайте .env у корінь проєкту."
    )

client = OpenAI()
EMBED_MODEL = "text-embedding-3-large"
CHAT_MODEL = "gpt-4o-mini"

DATA_DIR = PROJECT_ROOT / "data"
CHROMA_DIR = PROJECT_ROOT / "chroma_data" / "realtor_assistant"
COLLECTION_NAME = "realtor_corpus_v2"

print("=" * 60)
print(f"IN_COLAB: {IN_COLAB}")
print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"CHROMA_DIR: {CHROMA_DIR}")
print("=" * 60)

IN_COLAB: False
PROJECT_ROOT: /Users/nmana/projects/RAG-workshop_Day2
DATA_DIR: /Users/nmana/projects/RAG-workshop_Day2/data
CHROMA_DIR: /Users/nmana/projects/RAG-workshop_Day2/chroma_data/realtor_assistant


### Допоміжні функції: chunking і embeddings

Той самий підхід, що в дні 1: нарізка з **overlap**, щоб не «різати» важливі межі між чанками.

Для індексації додаємо **batch**-ембеддинги замість окремого ембеддингу для кожного чанку (менше HTTP-запитів).


In [22]:
# Розбиває довгий текст на шматки однакової довжини з перекриттям (overlap).
def chunker(text: str, chunk_size: int = 400, overlap: int = 80) -> list[str]:
    chunks, start = [], 0
    while start < len(text):
        piece = text[start : start + chunk_size].strip()
        if piece:
            chunks.append(piece)
        start += chunk_size - overlap
    return chunks


# Повертає векторні представлення для кількох текстів одним запитом до API ембеддингів.
def embed_many(texts: list[str]) -> list[list[float]]:
    if not texts:
        return []
    resp = client.embeddings.create(model=EMBED_MODEL, input=texts)
    return [item.embedding for item in resp.data]


### Для допитливих: що було до RAG-підходу

Колись подібні «асистенти по наших документах» часто робили як **окремий проєкт під задачу**: вручну або правилами **розбирати** PDF/договори на поля, проєктувати **SQL-схему**, форми й звіти. Додати новий тип документа або поле зазвичай означало **довгу розробку, міграції й підтримку парсерів** — дорого і повільно на старті й потім.

**Повнотекстовий пошук** (на кшталт інвертованого індексу в Elasticsearch) допомагав знаходити слова й фрази, але **семантичні** запити («що у нас написано про розірвання») без сучасних ембеддингів і моделей були скрутіші; усе одно потрібні були індексація й налаштування релевантності.

**RAG** не замінює класичну БД там, де потрібні транзакції та жорстка структура, але дає інший компроміс: **файли → індекс → запитання звичайною мовою** з меншою кількістю унікального коду «під кожен кейс». Підготовка даних, chunking і перевірка відповідей усе ще важливі — просто поріг входу для прототипу зазвичай **нижчий**, ніж будувати повноцінну предметну SQL-систему з нуля.


---
## 01 - Збір корпусу документів з `data/` — чотири важливих кроки

1. **Сканування:** обходимо всю папку `data/` (включно з підпапками), збираємо файли й визначаємо тип за розширенням (`.md`, `.csv`, `.pdf`).
2. Підготовка допоміжних **функцій за типом файлів:** для кожного формату — прочитати текст, за потреби нарізати на чанки, створити методанні, створити ідентифікаторі для індексацій. 
3. Створення **колекції** в **ChromaDB** для зберігання векторів і текстів чанків.
4. **Цикл:** для кожного файлу викликаємо відповідну функцію й завантажуємо чанки в колекцію Chroma (`upsert` батчами).


### Крок 1: Сканування - перегляд що ми маємо в каталозі даних. 

In [23]:
# --- Крок 1: що є у корені даних ---
SUPPORTED_SUFFIXES = {".md", ".csv", ".pdf"}


# Знаходить усі підтримувані файли в каталозі (рекурсивно, включно з підпапками).
def iter_corpus_files(root: Path) -> list[Path]:
    out: list[Path] = []
    if not root.is_dir():
        return out
    for path in sorted(root.rglob("*")):
        if path.is_file() and path.suffix.lower() in SUPPORTED_SUFFIXES:
            out.append(path)
    return out


DATA_FILES = iter_corpus_files(DATA_DIR)
print(f"Знайдено файлів: {len(DATA_FILES)}")
for p in DATA_FILES:
    rel = p.relative_to(PROJECT_ROOT)
    print(f"  {rel}  →  {p.suffix.lower()}")



Знайдено файлів: 9
  data/agency_buyer_timeline.md  →  .md
  data/clients.csv  →  .csv
  data/contracts/dogovir_Petrenko_rent.pdf  →  .pdf
  data/contracts/dogovir_Petrenko_sale.pdf  →  .pdf
  data/contracts/dogovir_svitlana_dorosh_L005.pdf  →  .pdf
  data/listings.csv  →  .csv
  data/mortgage_advisor_checklist.md  →  .md
  data/policy_short_term_rental.md  →  .md
  data/taxes_2026.md  →  .md


### Крок 2: Пофайлова обробка

#### Обробка різних типів файлів

- **`process_md`** / **`process_pdf`** — один файл → текст → `chunker` → списки `id`, текстів і метаданих.
- **`process_csv`** — **один рядок = один логічний документ** (як окремий «міні-досьє»); якщо рядок довгий, його також можна нарізати на кілька чанків.
- **`upsert_batches`** — спільна відправка в Chroma: ембеддинги батчами й **`upsert`** (ідемпотентно за стабільними `id`).

**Складнощі з PDF.** У цьому ноутбуку текст з PDF знімається через **PyMuPDF (`fitz`)** — це працює для «електронних» PDF із текстовим шаром (аналогічну задачу часто вирішують бібліотекою **pypdf**). Якщо ж документ — **скан** або зображення без нормального текстового шару, потрібні **OCR** (наприклад Tesseract або хмарні сервіси), часто ще й попередня обробка якості та посторінковий пайплайн — дорожче й повільніше.

**Особливості CSV.** У нас **один рядок таблиці = один логічний документ** для індексації (текст збирається з полів у читабельний блок). Зверніть увагу на **кодування** (краще UTF-8), **роздільники та лапки**, пропуски в колонках і те, що після зчитування значення зазвичай усі **рядкові** — семантичний пошук не замінює точну фільтрацію як у SQL. У спрощеній схемі метаданих лише **`source`** файлу; щоб фільтрувати за колонками (тип угоди, бюджет тощо), довелося б додавати поля в метадані або тримати таблицю окремо в БД.

#### Метадані

У метаданих зберігаємо лише **`source`** (відносний шлях файлу). **`where={"source": "…"}`** дозволяє шукати, наприклад, лише в `data/listings.csv` або лише в одному PDF з `data/contracts/`.

**Метадані в Chroma.** Крім **`source`** (шлях до файлу) інколи додають поля для фільтрації та цитування: **doc_type**, **page**, **chunk_index**, **record_id**, дати **updated_at**, або на якій мові **language**. У метадані варто класти лише те, по чому реально потрібен **`where`** або контроль доступу; решту зручніше лишати в тексті чанка.

#### Індексація

**Стабільні ID для Бази даних:** шлях файлу + номер рядка CSV (якщо є) + номер частини — щоб повторний запуск не створював дублікатів, це називається **Ідемпотентна індексація** - тобто для кожного чанка використовуються стабільні id, а операція upsert або створює запис, або перезаписує вже наявний з таким id.


#### Розширені метадані для Chroma (поза source)

У спрощеній схемі достатньо `source`, щоб обмежити пошук одним файлом. Для робочого асистента агентства цього може буде замало (залежить від набору запитів які мають буть оброблені): якщо часті запити — «все по клієнту C-101», «матеріали по L-005», «лише оренда в Старому Місті». Тоді в метадані додають бізнес-ключі (client_id, listing_id) і виміри фільтрації (тип угоди (sale/rent), район, статус (active/closed/paused), тип документа), а в текст чанка лишають деталі — суми, формулювання, коментарі, описи.

**Принцип**: у `meta` кладуть лише те, по чому реально потрібен фільтр, контроль доступу або зручне цитування (сторінка, номер чанка). Решта — у тексті документа. Поля можуть бути відсутні для частини корпусу: політики, чеклісти іпотеки, податкові довідки не прив’язані до клієнта чи об’єкта — глобальний пошук працює як раніше, а при відомому клієнті чи квартирі retrieval можна звузити додатковими полями.

In [24]:
# Перетворює відносний шлях файлу на безпечний префікс (заміняє "/" на "__" )для стабільних id у Chroma.
def _safe_id_base(rel_path: str) -> str:
    return rel_path.replace("/", "__").replace("\\", "__")


# Перетворює значення метаданих на рядки (Chroma приймає лише string values).
def _str_meta(meta: dict) -> dict:
    return {k: ("" if v is None else str(v)) for k, v in meta.items()}


# Зі списку чанків і одного словника метаданих будує списки id, текстів і метаданих для Chroma.
def _records_from_chunks(
    parts: list[str], id_prefix: str, meta: dict
) -> tuple[list[str], list[str], list[dict]]:
    ids, docs, metas = [], [], []
    for i, chunk in enumerate(parts):
        ids.append(f"{id_prefix}:part{i:04d}")
        docs.append(chunk)
        metas.append(_str_meta(dict(meta)))
    return ids, docs, metas


# Читає .md, нарізає на чанки; метадані мінімальні — лише source (шлях файлу для цитат і where).
def process_md(
    path: Path, chunk_size: int = 420, overlap: int = 90
) -> tuple[list[str], list[str], list[dict]]:
    text = path.read_text(encoding="utf-8").strip()
    parts = chunker(text, chunk_size=chunk_size, overlap=overlap)
    rel = str(path.relative_to(PROJECT_ROOT))
    base = _safe_id_base(rel)
    meta = {"source": rel}
    return _records_from_chunks(parts, base, meta)


# Витягує текст із PDF, нарізає на чанки; метадані лише source.
def process_pdf(
    path: Path, chunk_size: int = 420, overlap: int = 90
) -> tuple[list[str], list[str], list[dict]]:
    doc = fitz.open(path)
    try:
        pages_txt = [page.get_text() for page in doc]
    finally:
        doc.close()
    text = "\n".join(pages_txt).strip()
    text = re.sub(r"\n{3,}", "\n\n", text)
    parts = chunker(text, chunk_size=chunk_size, overlap=overlap)
    rel = str(path.relative_to(PROJECT_ROOT))
    base = _safe_id_base(rel)
    meta = {"source": rel}
    return _records_from_chunks(parts, base, meta)


# Кожен рядок CSV — окремий документ (з чанками); метадані лише source (усі рядки файлу мають той самий шлях).
def process_csv(
    path: Path, chunk_size: int = 420, overlap: int = 90
) -> tuple[list[str], list[str], list[dict]]:
    rel = str(path.relative_to(PROJECT_ROOT))
    base = _safe_id_base(rel)
    ids, docs, metas = [], [], []

    with path.open(encoding="utf-8", newline="") as f:
        for row_idx, row in enumerate(csv.DictReader(f)):
            lines = [f"# Рядок {row_idx + 1} — {path.name}", ""]
            for k, v in row.items():
                if v is None or str(v).strip() == "":
                    continue
                lines.append(f"- **{k}**: {v}")
            text = "\n".join(lines)
            parts = chunker(text, chunk_size=chunk_size, overlap=overlap)
            row_key = (
                row.get("record_id") or row.get("listing_id") or f"row{row_idx}"
            ).strip()
            row_key_safe = re.sub(r"[^\w\-]+", "_", row_key)
            prefix = f"{base}__row{row_idx:04d}_{row_key_safe}"
            meta = {"source": rel}
            ri, rd, rm = _records_from_chunks(parts, prefix, meta)
            ids.extend(ri)
            docs.extend(rd)
            metas.extend(rm)
    return ids, docs, metas


# Додає чанки в колекцію Chroma: рахує ембеддинги батчами й виконує upsert.
def upsert_batches(
    coll,
    ids: list[str],
    documents: list[str],
    metadatas: list[dict],
    batch_size: int = 64,
) -> None:
    if not ids:
        return
    for start in range(0, len(ids), batch_size):
        sl = slice(start, start + batch_size)
        embs = embed_many(documents[sl])
        coll.upsert(
            ids=ids[sl],
            documents=documents[sl],
            embeddings=embs,
            metadatas=metadatas[sl],
        )


### Крок 3. Створюємо місце для зберігання - ChromaDB: персистентний клієнт

**ChromaDB — топ‑функцій у цьому воркшопі:**

1. **`PersistentClient(path=...)`** — індекс на диску між перезапусками.
2. **`get_or_create_collection(name, metadata=...)`** — колекція чанків; `metadata={"hnsw:space": "cosine"}` задає метрику для **`distances`**.
3. **`collection.upsert(...)`** — завантаження/оновлення чанків за стабільними `id` (без дублікатів при повторній індексації).
4. **`collection.query(query_embeddings=..., n_results=k, where=..., include=[...])`** — dense retrieval, опційно `where={"source": "…"}`.
5. **`collection.count()`** / **`collection.peek(limit=...)`** — скільки чанків у колекції та швидкий перегляд після індексації
6. Додатково при перебудові:* **`delete_collection(name)`** — скинути стару колекцію перед **`FORCE_REBUILD`**.Якщо колекція вже непуста й `FORCE_REBUILD = False`, блок 4 можна пропустити — економія на ембеддингах. Якщо потрібно видалити колекцію й переіндексувати (наприклад, змінили модель ембеддингів або chunking) — встановіть **`FORCE_REBUILD = True`**.
7. Нагадування: `hnsw:space: cosine` — у результаті `query` поле **distances** — це cosine distance (менше = ближче).


In [ ]:
FORCE_REBUILD = False  # True — повна перебудова

CHROMA_DIR.mkdir(parents=True, exist_ok=True)
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))

if FORCE_REBUILD:
    try:
        chroma_client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

### Крок 4: цикл по обробці файлів і завантаження в Chroma

Для кожного шляху з `DATA_FILES` викликаємо відповідну функцію з блоку 2 і одразу **`upsert_batches`** у спільну колекцію.

Після індексації зручно перевірити **`collection.count()`** і **`collection.peek(limit=...)`** — перші записи в колекції (див. код-комірку нижче).


In [26]:
if FORCE_REBUILD or collection.count() == 0:
    print("Індексація: обхід файлів у data/ …")
    for path in DATA_FILES:
        suf = path.suffix.lower()
        if suf == ".md":
            ids, docs, metas = process_md(path)
        elif suf == ".csv":
            ids, docs, metas = process_csv(path)
        elif suf == ".pdf":
            ids, docs, metas = process_pdf(path)
        else:
            continue
        if not docs:
            print("  пропуск (порожній):", path.relative_to(PROJECT_ROOT))
            continue
        upsert_batches(collection, ids, docs, metas)
        print(f"  OK {path.relative_to(PROJECT_ROOT)} → {len(docs)} чанків")
    print("Готово. Чанків у колекції:", collection.count())
else:
    print("Колекція вже є; індексацію пропущено. Чанків:", collection.count())
    print("(для повної перебудови: FORCE_REBUILD = True)")


Індексація: обхід файлів у data/ …


  OK data/agency_buyer_timeline.md → 3 чанків
  OK data/clients.csv → 22 чанків
  OK data/contracts/dogovir_Petrenko_rent.pdf → 9 чанків
  OK data/contracts/dogovir_Petrenko_sale.pdf → 8 чанків
  OK data/contracts/dogovir_svitlana_dorosh_L005.pdf → 9 чанків
  OK data/listings.csv → 10 чанків
  OK data/mortgage_advisor_checklist.md → 4 чанків
  OK data/policy_short_term_rental.md → 3 чанків
  OK data/taxes_2026.md → 4 чанків
Готово. Чанків у колекції: 72


---
## 02 - Retrieval

Після індексації в колекції вже є **чанки**, їхні **ембеддинги** (той самий виклик **`embed_many`** і модель **`EMBED_MODEL`**, що під час **`upsert_batches`**) та **`source`** у метаданих.

Функція **`retrieve`** нижче реалізує типовий **dense retrieval**: рахує вектор запиту й викликає **`collection.query`** із **`query_embeddings`**, **`n_results=k`**, опційно **`where`** (фільтр по метаданих Chroma) та **`include=["documents", "distances", "metadatas"]`** — саме ці поля ми одразу друкуємо для дебагу. Колекція створена з **`metadata={"hnsw:space": "cosine"}`**, тому в **`distances`** значення **менші означають ближче** за косинусною метрикою. Це наближений пошук **найближчих сусідів** у просторі ембеддингів, а не класичний повнотекстовий індекс.

Спочатку буде запит **без** `where` по всій колекції, потім — приклади **`where={"source": "…"}`** лише для одного файлу.

In [27]:
# Повертає top-k найближчих чанків за косинусною відстанню (за потреби з фільтром метаданих where).
def retrieve(query: str, k: int = 6, where: dict | None = None):
    q_emb = embed_many([query])[0]
    return collection.query(
        query_embeddings=[q_emb],
        n_results=k,
        where=where,
        include=["documents", "distances", "metadatas"],
    )


q = "Які квартири в оренду в районі Старе Місто?"
res = retrieve(q, k=5)
for dist, doc, meta in zip(res["distances"][0], res["documents"][0], res["metadatas"][0]):
    print(f"[{dist:.3f}] {meta.get('source')}")
    print(doc[:280].replace("\n", " ") + ("…" if len(doc) > 280 else ""))
    print("---")


[0.451] data/policy_short_term_rental.md
# Короткострокова оренда в історичних кварталах (умовні правила ОСББ «Старе Місто»)  Корпус для воркшопу; реальні статути ОСББ завжди читати повністю.  ## Обмеження для Airbnb / добової оренди  У будинках під управлінням ОСББ зони **Старе Місто–ядро** короткострокова оренда **до …
---
[0.517] data/agency_buyer_timeline.md
годили верхню межу **4_800_000 грн** для двокімнатної у Старому Місті / Центрі. 3. **Юридична перевірка об’єкта** — виписка з реєстру, обтяження, статус продавця (за дорученням юриста партнерів). 4. **Умовна резервація / лист про наміри** — узгоджуємо ціну та строки до нотаріуса.…
---
[0.562] data/listings.csv
# Рядок 3 — listings.csv  - **listing_id**: L-003 - **type**: rent - **monthly_rent_uah**: 14200 - **area_m2**: 41 - **rooms**: 1 - **district**: Старе Місто - **status**: active - **address**: вул. Вірменська, 8, кв. 2 - **description**: Студія з окремою кухнею; свіжий ремонт; п…
---
[0.580] data/contracts/dogovir_svitla

In [28]:
q = "Які умови оплати в договорі з Петренко?"
res = retrieve(q, k=5)
for dist, doc, meta in zip(res["distances"][0], res["documents"][0], res["metadatas"][0]):
    print(f"[{dist:.3f}] {meta.get('source')}")
    print(doc[:280].replace("\n", " ") + ("…" if len(doc) > 280 else ""))
    print("---")

[0.501] data/contracts/dogovir_Petrenko_rent.pdf
у  договорі  оренди,  підписаному  Клієнтом  з  орендодавцем. 2.2. Оплата винагороди здійснюється Клієнтом після підписання договору оренди  квартири та підтвердження фактичної домовленості між Клієнтом і орендодавцем. 2.3. Оплата здійснюється на підставі акту наданих послуг або …
---
[0.511] data/contracts/dogovir_Petrenko_sale.pdf
янка  Петренко Марія Олегівна, паспорт серії XX №000000, ІПН 0000000000, іменується надалі «Клієнт», з іншого  боку, разом іменовані «Сторони», уклали цей Договір про нижченаведене: 1. ПРЕДМЕТ ДОГОВОРУ 1.1. Клієнт доручає, а Агентство приймає на себе зобов'язання за винагороду зд…
---
[0.515] data/clients.csv
оботі; торг за L-001 (кілька пропозицій до закриття угоди Івана); комісія за стандартною сіткою після підписання.
---
[0.519] data/clients.csv
ія 50% від першого місяця оренди за об'єктом L-002 (18_500 грн) + ПДВ за рахунком; загалом на кшталт 11_100 грн з ПДВ. - **last_agreement_note**: Річний договір п

### Фільтр `where` (лише за файлом — `source`)

У метаданих кожного чанка ми зберігаємо лише **`source`**: відносний шлях до файлу в проєкті (як у списку після сканування `data/`).

Тоді пошук можна обмежити одним джерелом:

- каталог оголошень: `where={"source": "data/listings.csv"}`
- один договір (PDF): `where={"source": "data/contracts/dogovir_Petrenko.pdf"}`

Якщо змінили код індексації або колекцію — зробіть повну перебудову (**`FORCE_REBUILD = True`**). Зараз у коді колекція **`realtor_corpus_v2`**, щоб не змішувати зі старими метаданими.


In [29]:
# Лише чанки з одного файлу — шлях має збігатися з тим, що в метаданих після індексації
res_f = retrieve(
    "Які квартири в оренду в районі Старе Місто?",
    k=5,
    where={"source": "data/listings.csv"},
)
for dist, doc, meta in zip(res_f["distances"][0], res_f["documents"][0], res_f["metadatas"][0]):
    print(f"[{dist:.3f}] {meta.get('source')} | {doc[:220]}…")


[0.562] data/listings.csv | # Рядок 3 — listings.csv

- **listing_id**: L-003
- **type**: rent
- **monthly_rent_uah**: 14200
- **area_m2**: 41
- **rooms**: 1
- **district**: Старе Місто
- **status**: active
- **address**: вул. Вірменська, 8, кв. 2
…
[0.595] data/listings.csv | # Рядок 2 — listings.csv

- **listing_id**: L-002
- **type**: rent
- **monthly_rent_uah**: 18500
- **area_m2**: 54
- **rooms**: 2
- **district**: Старе Місто
- **status**: active
- **address**: пл. Ринок, 3, кв. 15
- **d…
[0.600] data/listings.csv | - **pets_allowed**: cats_ok
- **hoa_uah_month**: 1200
- **parking**: courtyard
- **school_note**: Гімназія ім. Франка — 12 хв.…
[0.619] data/listings.csv | # Рядок 1 — listings.csv

- **listing_id**: L-001
- **type**: sale
- **asking_price_uah**: 5200000
- **area_m2**: 68
- **rooms**: 2
- **district**: Старе Місто
- **status**: active
- **address**: вул. Домініканська, 12, …
[0.634] data/listings.csv | : dogs_under_10kg_only
- **hoa_uah_month**: 2100
- **parking**: un

In [30]:
# Приклад: користувач просить опиратися лише на текст конкретного договору (PDF)
CONTRACT_SOURCE = "data/contracts/dogovir_Petrenko_sale.pdf"

res_contract = retrieve(
    "Які умови оплати в договорі з Петренко?",
    k=5,
    where={"source": CONTRACT_SOURCE},
)
for dist, doc, meta in zip(
    res_contract["distances"][0],
    res_contract["documents"][0],
    res_contract["metadatas"][0],
):
    print(f"[{dist:.3f}] {meta.get('source')} | {doc[:220]}…")


[0.511] data/contracts/dogovir_Petrenko_sale.pdf | янка 
Петренко Марія Олегівна, паспорт серії XX №000000, ІПН 0000000000, іменується надалі «Клієнт», з іншого 
боку, разом іменовані «Сторони», уклали цей Договір про нижченаведене:
1. ПРЕДМЕТ ДОГОВОРУ
1.1. Клієнт доруча…
[0.536] data/contracts/dogovir_Petrenko_sale.pdf | ез письмової згоди Агентства.
2. ВИНАГОРОДА ТА ПОРЯДОК РОЗРАХУНКІВ
2.1. Розмір винагороди Агентства становить 2,5% (дві цілих п'ять десятих відсотка) від фактичної ціни 
продажу,  зазначеної  в  нотаріально  посвідченому…
[0.539] data/contracts/dogovir_Petrenko_sale.pdf | никах, що мають однакову юридичну силу, по одному для кожної зі 
Сторін.
5.2. Зміна умов - лише у письмовій формі, підписаній обома Сторонами.
Підписи Сторін:
Директор ТОВ «Агентство Нерухомості Альфа»
Клієнт
___________…
[0.552] data/contracts/dogovir_Petrenko_sale.pdf | ротягом 5 (п'яти) банківських днів з дати державної реєстрації переходу права 
власності на користь покупця, на підставі акту нада

---
## 03 - Augmentation + Generation

### Промпт-інжиніринг для RAG

**Що це:** проєктування **системної поведінки** моделі в ланцюгу «retrieval → генерація»: які інструкції, який контекст і в якому порядку вона отримує, щоб відповідь була **релевантною, обмеженою фактами з бази знань і передбачуваною за форматом**.

**Чим відрізняється від «звичайного» промптингу:** модель **не має доступу** до всієї бази — лише до **відібраних фрагментів** і правил. Промпт має одночасно керувати **тим, як читати контекст**, **коли відмовлятися**, **як цитувати** і **що робити при слабкому retrieval** (мало чанків, низька релевантність, суперечливі джерела).

**Типові шари промпта в RAG:**

| Шар | Навіщо |
|-----|--------|
| **Роль і межі** | Хто асистент, для кого, який тон і що поза скоупом |
| **Правила роботи з контекстом** | Лише з наданих фрагментів; явна відмова «даних немає»; не змішувати джерела без потреби |
| **Формат відповіді** | Структура, мова, довжина, блок цитувань / посилань |
| **Політика джерел** | Що вважати «джерелом» (файл, сторінка, URL), як перелічувати, без вигаданих посилань |
| **Обробка невизначеності** | Що робити при порожньому або суперечливому контексті |
| **Few-shot** | 1–2 зразки «питання + контекст → відповідь» для стабільного стилю |
| **Окремі промпти в пайплайні** | Query rewriting, маршрутизація колекцій, re-rank hints, оцінка «чи достатньо контексту» — не один монолітний текст |

**Що промпт не замінює:** якість індексу, chunking, retrieval, reranking, фільтри метаданих і **перевірки в коді** (allowlist джерел, ліміти контексту, eval на golden-set). Промпт **доповнює** інженерію даних, а не компенсує поганий пошук.

**Практичний принцип:** спочатку зробити retrieval **достатньо точним**, потім уточнювати промпт під **відмови, цитування й формат**; інакше модель «красиво вигадує» там, де контекст слабкий.




### Кейс 1 (базовий):

**Промпт:**

- Основа з роллю, межами, обмеження галюцінації інструкції для вклю + виклики `rag_answer` **без** `where` — retrieval по всьому корпусу.

- Інструкції для включення **«Джерела:»** як список рядків **`[source=…]`** (шлях до файлу, як у заголовках контексту), включно з CSV — без заміни на **record_id** з тексту рядка. 

**Retrival** за замовчуванням виконує тільки пошук по embeddings

In [31]:
PROMPT_TEMPLATE = """Ти внутрішній асистент ріелтора агентства. Відповідай українською.
Правила:
- Спирайся лише на наведений контекст. Якщо даних недостатньо, або вони відсутні — скажи прямо.
- У кінці додай блок **«Джерела:»**: маркований список рядків **точно у форматі заголовків контексту** — `[source=…]`, де шлях такий самий, як у рядку перед фрагментом (наприклад `[source=data/clients.csv]`).
- Якщо факт узятий з фрагмента таблиці CSV, джерело все одно вказуй як **файл** у квадратних дужках (`data/listings.csv`, `data/clients.csv` тощо). **Не** вказуй замість джерела внутрішні поля з тексту рядка (**record_id**, **listing_id**, **client_id** тощо) — це не шлях до файлу.
- Переліч лише ті `[source=…]`, на які реально спиралась відповідь (без повторів).

Контекст:
{context}

Питання: {question}
Відповідь:"""


# Перетворює результат retrieval на один текст «контекст» із позначками джерел для промпта.
def format_hits(res: dict) -> str:
    blocks = []
    for doc, meta in zip(res["documents"][0], res["metadatas"][0]):
        hdr = f"[source={meta.get('source')}]"
        blocks.append(hdr + "\n" + doc)
    return "\n\n---\n\n".join(blocks)


# Повний RAG: пошук чанків, збірка промпту й відповідь чат-моделі.
def rag_answer(
    question: str,
    k: int = 6,
    where: dict | None = None,
    history: list[dict] | None = None,  # опційно: попередні user/assistant репліки
) -> str:
    res = retrieve(question, k=k, where=where)
    context = format_hits(res)
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)
    messages = list(history or [])  # історія діалогу перед поточним RAG-запитом
    messages.append({"role": "user", "content": prompt})
    out = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=messages,
        temperature=0.2,
    )
    return out.choices[0].message.content

In [32]:
# Quick test for case 1: 
print("\n===\n  RAG answers:")
print(rag_answer("Який відсоток податку на купівлю житла у 2026 для першої угоди в матеріалах?",))    
    
### Питання для самостійної перевірки. Яке з цих питань є перевіркою на галюцінацію?

#"Що в нотатках про домовленості з Іваном Коваленком щодо бюджету?",
#"Який перелік документів для зустрічі з кредитним радником?",
#"Які квартири є в оренду в районі Старе Місто?",
#"Яка мінімальна зарплата необхідна для отримання кредиту?",
#"Покажи закриті угоди по всіх клієнтах .",
#"Які умови по оплаті в договорі з Петренко?"



===
  RAG answers:
У 2026 році для першої купівлі єдиного житла фізособою в межах пільгової площі діє ставка **1%** від оціночної вартості або ціни договору (береться більша з дозволених підстав за методикою банку).

**Джерела:**
- [source=data/taxes_2026.md]


### Кейс 2: маршрутизація `where` через промпт

У **Кейсі 1** використовується звичайний `rag_answer(...)` **без** `where` — retrieval по всьому корпусу.

Тут додається **крок роутера**: невеликий промпт дає моделі **список дозволених шляхів** (`source`), збіг яких із тим, що реально проіндексовано з `DATA_FILES`. Модель повертає **один або кілька шляхів** (по рядку або через кому), або **`null`**, якщо обмежувати пошук не потрібно.

У коді відповідь **парситься і фільтрується по allowlist**; для кількох файлів у `where` — **`{"source": {"$in": [...]}}`**. Якщо жоден шлях не пройшов перевірку — retrieval **без** `where`.


In [33]:
# Ті самі відносні шляхи, що потрапили в індекс (унеможливлює вигадані source у відповіді моделі)
ALLOWED_SOURCES = sorted({str(p.relative_to(PROJECT_ROOT)) for p in DATA_FILES})

ROUTER_PROMPT = """Ти маєш розпізнати, які файли зі списку потрібні для відповіді на питання.
Дозволені шляхи source (обирай лише з цього списку):
{sources_list}

Питання користувача: {question}

Підказки:
- каталог квартир / оголошення → data/listings.csv
- клієнти, угоди, записи про людей → data/clients.csv
- договір, contracts + прізвище латиницею → один або кілька PDF з data/contracts/
  (якщо питання про всі договори клієнта без уточнення оренда/купівля — поверни всі його PDF)
- загальні правила (податки, політики, чеклисти .md) без явної вказівки «лише цей файл» → null

Відповідь ТІЛЬКИ шляхами зі списку, без пояснень і без markdown:
- один файл — один рядок з точним шляхом
- кілька файлів — по одному шляху на рядок (або через кому в одному рядку)
- якщо обмежувати пошук не потрібно — один рядок: null
"""


def _parse_router_sources(raw: str) -> list[str]:
    """Витягує з відповіді роутера лише шляхи з ALLOWED_SOURCES (порядок збережено)."""
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    if raw.lower() in {"null", "none", ""}:
        return []

    allowed = set(ALLOWED_SOURCES)
    picked: list[str] = []
    seen: set[str] = set()

    for line in raw.splitlines():
        for part in line.split(","):
            token = part.strip().strip("\"'")
            if token.lower() in {"null", "none", ""}:
                continue
            if token in allowed and token not in seen:
                picked.append(token)
                seen.add(token)

    return picked


def where_from_sources(sources: list[str] | None) -> dict | None:
    """Один source — рівність; кілька — Chroma $in."""
    if not sources:
        return None
    if len(sources) == 1:
        return {"source": sources[0]}
    return {"source": {"$in": sources}}


# Проаналізувавши запит користувача LLM повертає список дозволених source або None (retrieval без where).
def choose_sources_for_query(question: str) -> list[str] | None:
    sources_lines = "\n".join(f"- {s}" for s in ALLOWED_SOURCES)
    prompt = ROUTER_PROMPT.format(sources_list=sources_lines, question=question)
    out = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    raw = (out.choices[0].message.content or "").strip()
    picked = _parse_router_sources(raw)
    return picked if picked else None

# Кейс 2: Роутер + RAG
def rag_answer_with_router(question: str, k: int = 6) -> str:
    picked = choose_sources_for_query(question)
    where = where_from_sources(picked)
    res = retrieve(question, k=k, where=where)
    context = format_hits(res)
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)
    out = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,
    )
    return out.choices[0].message.content


In [34]:
question="Які умови по оплаті в договорі з Петренко?"
# без додаткової маршрутизації до конкретного джерела 

print("\n===\n", question)
print("\n--- Case 1: No router\n")
print(rag_answer(question))
print("\n--- Case 2: Router\n")
# з додатковою маршрутизацією до конкретного джерела 
print(rag_answer_with_router(question))
print("\n---\n")


===
 Які умови по оплаті в договорі з Петренко?

--- Case 1: No router

У договорі з Петренко зазначено, що оплата винагороди здійснюється Клієнтом після підписання договору оренди квартири та підтвердження фактичної домовленості між Клієнтом і орендодавцем. Розмір винагороди Агентства становить 50% від розміру місячної орендної плати, зазначеної у договорі оренди. Оплата здійснюється на підставі акту наданих послуг або іншого документа, що підтверджує надання послуг, якщо інше письмово не погоджено Сторонами.

**Джерела:**
- [source=data/contracts/dogovir_Petrenko_rent.pdf]

--- Case 2: Router

У договорі з Петренко зазначені такі умови по оплаті:

1. У договорі оренди: 
   - Оплата винагороди здійснюється після підписання договору оренди квартири та підтвердження фактичної домовленості між Клієнтом і орендодавцем.
   - Розмір винагороди Агентства становить 50% від розміру місячної орендної плати, зазначеної у договорі оренди.
   - Оплата здійснюється на підставі акту наданих послуг 

### Keйс 3:  памятаємо діалог
#### Історія чату в RAG
**Історія** — це ланцюжок останніх реплік (user / assistant) у межах однієї сесії (у Telegram — за chat_id). Вона допомагає зрозуміти уточнення («той самий Петренко», «а в оренді?»), але не замінює пошук у Chroma: факти з PDF і CSV як і раніше беруться через retrieval за поточним питанням, а історію додають у промпт поруч із знайденими чанками.

Зберігають у **пам’яті процесу** (`dict[chat_id]` → список повідомлень) або в SQLite/Redis для постійності. Щоб не перевищити ліміт токенів, зазвичай лишають вікно — останні 3–5 обмінів; в історію кладуть текст відповіді, а не весь RAG-контекст. Після кожної відповіді дописують пару user + assistant і передають її в наступний виклик messages разом із свіжим блоком «Контекст з бази».

In [35]:
# chat_id → список {"role": "user"|"assistant", "content": "..."}
CHAT_HISTORY: dict[int, list[dict]] = {}
MAX_HISTORY_TURNS = 4  # скільки пар user+assistant лишати


def trim_history(messages: list[dict], max_turns: int = MAX_HISTORY_TURNS) -> list[dict]:
    """Лишає останні max_turns пар (2 * max_turns повідомлень)."""
    return messages[-(2 * max_turns) :]


def get_history(chat_id: int) -> list[dict]:
    return CHAT_HISTORY.get(chat_id, [])


def append_history(chat_id: int, question: str, answer: str) -> None:
    msgs = CHAT_HISTORY.get(chat_id, [])
    msgs = trim_history(msgs + [
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ])
    CHAT_HISTORY[chat_id] = msgs


# Кейс 3: роутер + RAG + історія (повний ланцюжок)
def rag_answer_with_router_history(
    question: str,
    k: int = 6,
    chat_id: int | None = None,  # [зміна] None = без збереження; int = сесія діалогу
) -> str:
    history = get_history(chat_id) if chat_id is not None else None  # [зміна] читаємо історію сесії
    picked = choose_sources_for_query(question)
    where = where_from_sources(picked)
    res = retrieve(question, k=k, where=where)
    context = format_hits(res)
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)
    messages = list(history or [])  # [зміна] попередні репліки перед поточним RAG-запитом
    messages.append({"role": "user", "content": prompt})
    out = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=messages,
        temperature=0.2,
    )
    answer = out.choices[0].message.content
    if chat_id is not None:
        append_history(chat_id, question, answer)  # [зміна] зберігаємо пару питання/відповідь
    return answer


In [36]:
# Приклад діалогу з історією (фіктивний chat_id для ноутбука)
DEMO_CHAT_ID = 9001


def run_dialog(turns: list[str], chat_id: int = DEMO_CHAT_ID) -> None:
    CHAT_HISTORY.pop(chat_id, None)  # чиста сесія перед демо
    for i, q in enumerate(turns, 1):
        print(f"\n{'=' * 60}\nTurn {i}: {q}\n{'=' * 60}")
        print(rag_answer_with_router_history(q, k=5, chat_id=chat_id))


# Ланцюжок: спочатку договори Петренко, потім уточнення лише по продажу
run_dialog([
    "Які умови по оплаті в договорах з Петренко?",
    "тільки по продажам",
])



Turn 1: Які умови по оплаті в договорах з Петренко?
У договорах з Петренко встановлені такі умови по оплаті:

1. **Договір оренди**:
   - Оплата винагороди здійснюється Клієнтом після підписання договору оренди квартири та підтвердження фактичної домовленості між Клієнтом і орендодавцем.
   - Розмір винагороди Агентства становить 50% від розміру місячної орендної плати, зазначеної у договорі оренди.
   - Оплата здійснюється на підставі акту наданих послуг або іншого документа, що підтверджує надання послуг, якщо інше письмово не погоджено Сторонами.

2. **Договір купівлі-продажу**:
   - Розмір винагороди Агентства становить 2,5% від фактичної ціни продажу, зазначеної в нотаріально посвідченому договорі купівлі-продажу, але не менше ніж 80 000 грн.
   - Оплата здійснюється протягом 5 банківських днів з дати державної реєстрації переходу права власності.

**Джерела:**
- [source=data/contracts/dogovir_Petrenko_rent.pdf]
- [source=data/contracts/dogovir_Petrenko_sale.pdf]

Turn 2: тільки 

## Telegram-бот (опційно)

Потрібен пакет **`python-telegram-bot`**. Токен: **`TELEGRAM_TOKEN`** або **`TELEGRAM_BOT_TOKEN`** у `.env`.

Перед запуском бота виконайте комірки **роутера (Кейс 2)** і **історії (Кейс 3)** — бот викликає **`rag_answer_with_router_history`** з `chat_id` Telegram-чату; історія зберігається в **`CHAT_HISTORY`** у пам’яті процесу.

Команди: **`/reset`** — очистити історію діалогу в цьому чаті.


In [37]:
import os

from telegram.ext import ApplicationBuilder, CommandHandler, MessageHandler, filters

BOT_TOKEN = os.getenv("TELEGRAM_TOKEN") or os.getenv("TELEGRAM_BOT_TOKEN")

if not BOT_TOKEN:
    print("⚠️  TELEGRAM_TOKEN не встановлений. Бот не буде запущено.")
    print("💡 На Colab: додайте TELEGRAM_TOKEN до Secrets (🔑)")
    print("💡 Локально: додайте TELEGRAM_TOKEN до .env")
else:
    # Якщо комірка запускається повторно, коректно зупиняємо попередній інстанс.
    if "app" in globals() and app is not None:
        try:
            await app.updater.stop()
        except Exception:
            pass
        try:
            await app.stop()
        except Exception:
            pass
        try:
            await app.shutdown()
        except Exception:
            pass

    async def cmd_reset(update, context):
        """Очищує історію CHAT_HISTORY для цього чату."""
        chat_id = update.effective_chat.id
        CHAT_HISTORY.pop(chat_id, None)
        await update.message.reply_text("Історію діалогу очищено. Можете питати знову.")

    async def handle(update, context):
        if not update.message or not update.message.text:
            return
        chat_id = update.effective_chat.id
        question = update.message.text.strip()
        # Роутер + RAG + історія: get_history → LLM → append_history (Кейс 3)
        try:
            answer = rag_answer_with_router_history(question, k=5, chat_id=chat_id)
            await update.message.reply_text(answer)
        except Exception as e:
            await update.message.reply_text(f"Помилка: {str(e)[:200]}")

    app = ApplicationBuilder().token(BOT_TOKEN).build()
    app.add_handler(CommandHandler("reset", cmd_reset))
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, handle))

    await app.initialize()
    await app.start()
    await app.updater.start_polling()
    print("✅ Telegram-бот запущено (з історією по chat_id). /reset — очистити діалог.")
    print("💡 Зупинка: запустіть комірку нижче або перезавантажте Colab")

✅ Telegram-бот запущено (з історією по chat_id). /reset — очистити діалог.
💡 Зупинка: запустіть комірку нижче або перезавантажте Colab


In [38]:
# Зупинка бота (запустіть тільки якщо він працює)
if "app" in globals() and app is not None and BOT_TOKEN:
    try:
        await app.updater.stop()
        await app.stop()
        await app.shutdown()
        print("✅ Telegram-бот зупинено")
    except Exception as e:
        print(f"⚠️  Помилка при зупинці бота: {e}")
else:
    print("Бот не запущено або токен відсутній")

✅ Telegram-бот зупинено


## Evaluation та доналаштування RAG-систем

**Evaluation (оцінка)** — це перевірка, чи RAG **дійсно працює** на ваших даних і питаннях, а не лише «відповідає красиво». Після першого прототипу зазвичай збирають **набір тестових питань** (golden set): реальні запити користувачів або типові сценарії з `use_case_realtor_assistant.md`. Для кожного питання фіксують, **які фрагменти мали потрапити в контекст** і **яка відповідь прийнятна** (факти, формат, блок «Джерела:»).

**Як оцінюють на практиці:**
- **Retrieval** — чи в top‑k є потрібний файл/чанк (hit@k), чи не змішуються схожі договори чи CSV-рядки.
- **Відповідь** — чи факти з контексту, чи немає вигадок, чи джерела вказані коректно (`[source=…]`).
- Інколи — **end-to-end**: одне питання → повний ланцюг retrieve + generate, перевірка людиною або простими правилами в коді.

**Доналаштування (tuning)** починають, коли evaluation показує **системну** проблему, а не один випадковий промах:

| Симптом | Що зазвичай крутять спочатку |
|--------|------------------------------|
| Не той фрагмент у top‑k | **`k`**, **`chunk_size`**, **`overlap`**, потім reranker або гібрид BM25 + dense |
| Зайві джерела в контексті | **`where`** по `source`, роутер (як у Кейсі 2), окремі колекції за типом документа |
| Відповідь не з контексту | промпт (правила, «Джерела:»), температура моделі |
| Після змін коду/даних | **`FORCE_REBUILD = True`** і нова колекція (у нас — `realtor_corpus_v2`) |

Цикл виглядає так: **змінили параметр → переіндексували (за потреби) → знову прогнали golden set → порівняли**. Так поступово підвищують якість, не «вгадуючи» одне поле за раз без метрик.



In [39]:
# Приклад простого датасету для евалюації
EVAL_SET = [
  {
    "id": "E01",
    "question": "Який відсоток податку на першу купівлю житла у 2026?",
    "expected_sources": ["data/taxes_2026.md"],
    "final_answer": "У 2026 році для першої купівлі єдиного житла фізособою в межах пільгової площі діє ставка 1% від оціночної вартості або ціни договору (береться більша з дозволених підстав за методикою банку)."
  },
  {
    "id": "E02",
    "question": "Які умови по оплаті в договорі з Петренко?",
    "expected_sources": [
      "data/contracts/dogovir_Petrenko_rent.pdf",
      "data/contracts/dogovir_Petrenko_sale.pdf"
    ],
    "final_answer": "Договір оренди: винагорода агентства 50% від місячної орендної плати після підписання договору оренди та підтвердження домовленості з орендодавцем; оплата за актом наданих послуг. Договір купівлі-продажу: винагорода 2,5% від фактичної ціни продажу в нотаріальному договорі, але не менше 80 000 грн; оплата протягом 5 банківських днів після державної реєстрації переходу права власності, за актом наданих послуг."
  },
  {
    "id": "E03",
    "question": "Які квартири в оренду в районі Старе Місто?",
    "expected_sources": ["data/listings.csv"],
    "final_answer": "У каталозі в оренду в районі Старе Місто: L-002 (2 кімнати, 18 500 грн/міс., пл. Ринок, 3) та L-003 (студія, 14 200 грн/міс., вул. Вірменська, 8); обидва зі статусом active."
  }
]


In [40]:
import re

# За замовчуванням rag_fn=rag_answer (Кейс 1); для роутера: rag_fn=rag_answer_with_router
JUDGE_PROMPT = """Ти суддя якості RAG. Чи відповідь ACTUAL містить ті самі ключові факти, що й REFERENCE (допускається інше формулювання)?
Відповідь лише одним словом: YES або NO.

Питання: {question}

REFERENCE:
{reference}

ACTUAL:
{actual}
"""


def extract_sources(answer: str) -> set[str]:
    """Витягує шляхи з блоку «Джерела:» у форматі [source=…] з тексту відповіді."""
    return set(re.findall(r"\[source=([^\]]+)\]", answer or ""))


def judge_matches_reference(question: str, reference: str, actual: str) -> bool:
    """LLM-judge: чи ACTUAL за фактами еквівалентна еталонній REFERENCE (final_answer)."""
    prompt = JUDGE_PROMPT.format(question=question, reference=reference, actual=actual)
    out = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return (out.choices[0].message.content or "").strip().upper().startswith("YES")


def run_eval(cases: list[dict], rag_fn=rag_answer, k: int = 5) -> list[dict]:
    """Проганяє EVAL_SET: RAG-відповідь, source coverage і judge по final_answer; друкує звіт."""
    results = []
    for case in cases:
        actual = rag_fn(case["question"], k=k)
        found = extract_sources(actual)
        expected = set(case["expected_sources"])
        row = {
            "id": case["id"],
            "sources_ok": expected <= found,  # усі очікувані файли процитовані
            "found_sources": sorted(found),
            "answer_ok": judge_matches_reference(case["question"], case["final_answer"], actual),
            "actual": actual,
        }
        print("Питання: ", case["question"])
        print("Знайдені джерела: ", found)
        print("Очікуємо джерела: ", expected)
        print("Отримана відповідь: ", actual)
        results.append(row)
        mark = "✓" if row["sources_ok"] and row["answer_ok"] else "✗"
        print(f"{mark} {case['id']}: sources={row['sources_ok']} answer={row['answer_ok']}")
        print("--------------------------------")
    passed = sum(1 for r in results if r["sources_ok"] and r["answer_ok"])
    print(f"\n{passed}/{len(results)} passed")
    return results


eval_results = run_eval(EVAL_SET)


Питання:  Який відсоток податку на першу купівлю житла у 2026?
Знайдені джерела:  {'data/taxes_2026.md'}
Очікуємо джерела:  {'data/taxes_2026.md'}
Отримана відповідь:  Відсоток податку на першу купівлю житла у 2026 році становить **1%** від оціночної вартості або ціни договору, беручи більшу з дозволених підстав за методикою банку.

**Джерела:**
- [source=data/taxes_2026.md]
✓ E01: sources=True answer=True
--------------------------------
Питання:  Які умови по оплаті в договорі з Петренко?
Знайдені джерела:  {'data/contracts/dogovir_Petrenko_rent.pdf'}
Очікуємо джерела:  {'data/contracts/dogovir_Petrenko_sale.pdf', 'data/contracts/dogovir_Petrenko_rent.pdf'}
Отримана відповідь:  У договорі з Петренко зазначено, що оплата винагороди здійснюється Клієнтом після підписання договору оренди квартири та підтвердження фактичної домовленості між Клієнтом і орендодавцем. Розмір винагороди Агентства становить 50% від розміру місячної орендної плати, зазначеної у договорі оренди. Оплата здійснює

#### Більш "дата саейнс" метрики evaluation для RAG
**На етапі Retrieval:** перевіряють, чи пошук приносить потрібні фрагменти — **Hit@k** (чи є релевантний чанк у top‑k), **Recall@k / Precision@k** (чим вище релевантний результат, тим краще). **Relevance** — наскільки чанк близький до запиту (косинус ембеддингів або окрема rerank-модель). **Source coverage** — чи всі очікувані файли з golden-set потрапили в цитати або в top‑k (|cited ∩ expected| / |expected|).

**На етапі генерації фінальної відповіді:** **groundedness / faithfulness** — чи факти в відповіді випливають із наданого контексту (менше галюцинацій); **correctness** — збіг із еталоном final_answer (F1, semantic similarity або LLM-judge). Groundedness не гарантує повноту: модель може чесно відповісти лише з одного PDF. Для воркшопу достатньо golden-set + coverage джерел + judge по final_answer; Hit@k і groundedness — коли порівнюєте зміни chunking, k або роутера на одному й тому ж EVAL_SET.

### Загальна схема RAG (цей воркшоп)

![Пайплайн: індексація, обробка запиту (Router, Where, History), evaluation](https://raw.githubusercontent.com/NatalyUA/RAG-workshop_Day2/main/AdvancedRAG_pipeline.png)

*На Colab: зображення завантажується з GitHub. Локально: файл у корені проєкту.*

---

### Куди рухаються далі (поза базовим воркшопом)

| Напрям | Що це | Приклади для ріелторського кейсу |
|--------|--------|----------------------------------|
| **RAG+** | Той самий ланцюг retrieve → generate, але **розумніший пошук і відбір контексту** | Гібрид BM25 + dense, re-ranking, окремі колекції (оголошення / клієнти / договори), query rewriting, HyDE. |
| **Мультимодальний RAG** | У індекс потрапляють не лише **текст**, а й **зображення** (ембеддинги картинок + текст) | Плани квартир, фото з оголошень, скани підписаних сторінок договору; запит «схожа планування» або «що на цій схемі». Потрібні OCR/vision-моделі та окремий пайплайн індексації. |
| **Агент / agentic RAG** | Модель **не один раз** відповідає, а **планує кроки**: обрати джерело, викликати пошук, перевірити контекст, за потреби повторити | Роутер `where` (Кейс 2) — найпростіший крок; далі — інструменти: `search_listings`, `read_contract`, перевірка «чи достатньо контексту», кілька раундів retrieval. Telegram-бот — **інтерфейс**, агентність — у **логіці всередині** (хто вирішує *що* шукати і *коли*). |
| **Дані та індекс** | Якість відповіді залежить від **корпусу і chunking**, не лише від промпта | OCR для сканів PDF, Excel, синхронізація з OLX, дедуплікація, версії документів, PII і ACL у метаданих. |
| **Операційка** | Як **тримати** систему в продакшені | Інкрементальний upsert, моніторинг, golden-set, A/B промптів. |
| **Поза RAG** | Задачі, де потрібні **тули** - тобто **транзакції, дії в світі**, а не тільки цитування документів | CRM, календар показів, скоринг лідів, прогноз ціни — окремі сервіси; RAG лишається відповіддю на «що написано в наших матеріалах». |
